# Feature Engineering — EV Purchase Prediction (S6E9)

**Competition:** Kaggle Playground Series S6E9 — predict whether a person will buy an electric vehicle.
**Goal:** Build all features from raw data and save to `data/processed/` for model training.

*Analysis was done independently. Code was reviewed and cleaned up with Claude Sonnet 4.6.*

---

**Feature groups built in this notebook:**

| Group | Features | Count |
|---|---|---|
| Numeric (raw) | Age, income, commute, etc. | 7 |
| Categorical (encoded) | Gender, city type, car type, binary flags | 6 |
| Manual scores | `buy_score`, `worry_score` | 2 |
| Interaction features | Cross-products of key variables | 7 |

**Total: 22 features**


## 1. Imports

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import LabelEncoder

# Fix working directory so relative paths work regardless of where Jupyter is launched
os.chdir(r'C:\Users\89671\Documents\Hackathons\kaggle-competitions\predicting_electric_vehicle_purchases_s6e9')

DATA = Path('data/raw')
PROC = Path('data/processed')

SEED = 42
np.random.seed(SEED)

print("Imports OK")
print(f"Working directory: {os.getcwd()}")


## 2. Load Data

In [ ]:
train = pd.read_csv(DATA / 'train.csv')
test  = pd.read_csv(DATA / 'test.csv')

# Lowercase all column names for consistency
train.columns = train.columns.str.lower()
test.columns  = test.columns.str.lower()

# Binary target: 1 = will buy EV, 0 = will not
y = (train['will_buy_ev'].astype(str) == 'Yes').astype(int)

print(f"Train: {train.shape},  Test: {test.shape}")
print(f"Target: {y.mean():.4f} positive rate  ({y.sum():,} / {len(y):,})")


## 3. Manual Features

Two hand-crafted scores based on patterns found during EDA.

**`buy_score`** — linear combination of the strongest purchase drivers.
Coefficients are derived from the original 10K-row dataset (Chris Deotte's analysis).
Higher score → person is more likely to buy an EV.

**`worry_score`** — how inconvenient EV charging would be for this person.
Long commute raises concern; home charging and nearby stations reduce it.
Higher score → more practical friction around owning an EV.


In [ ]:
def build_manual_features(df):
    df = df.copy()

    # buy_score: income, environmental concern, and subsidy push toward buying;
    # range anxiety (especially high) strongly pulls away from it
    df['buy_score'] = (
        1.2 * df['annual_income_usd'] / 1e5
        + 0.6 * df['environmental_concern_level']
        + 2.0 * (df['subsidy_available'].astype(str) == 'Yes').astype(int)
        - 1.0 * (df['range_anxiety_level'].astype(str) == 'Medium').astype(int)
        - 3.0 * (df['range_anxiety_level'].astype(str) == 'High').astype(int)
    )

    # worry_score: long commute is bad; home charging and nearby stations are good
    df['worry_score'] = (
        df['daily_commute_km']
        - 5.0  * df['charging_stations_near_home']
        - 5.0  * df['charging_stations_near_work']
        - 150.0 * (df['home_charging_possible'].astype(str) == 'Yes').astype(int)
    )

    return df


train = build_manual_features(train)
test  = build_manual_features(test)

print("buy_score and worry_score added")
print(f"buy_score  — mean: {train['buy_score'].mean():.2f},  std: {train['buy_score'].std():.2f}")
print(f"worry_score — mean: {train['worry_score'].mean():.2f},  std: {train['worry_score'].std():.2f}")


## 4. Interaction Features

Cross-products capture non-linear relationships that a linear model would miss.
LightGBM can find interactions on its own, but explicit features help it find them faster
and give the model more direct signal.


In [ ]:
def build_interactions(df):
    df = df.copy()

    # Financial capacity when subsidy is available — amplifies the subsidy effect
    df['income_x_subsidy'] = (
        df['annual_income_usd'] / 1e5
        * (df['subsidy_available'].astype(str) == 'Yes').astype(int)
    )

    # Environmental motivation combined with subsidy — doubly motivated buyers
    df['concern_x_subsidy'] = (
        df['environmental_concern_level']
        * (df['subsidy_available'].astype(str) == 'Yes').astype(int)
    )

    # Income per car slot — wealthier household relative to number of cars
    df['income_per_car'] = df['annual_income_usd'] / (df['number_of_cars_owned'] + 1)

    # Commute burden vs available charging infrastructure
    df['commute_per_charger'] = (
        df['daily_commute_km']
        / (df['charging_stations_near_home'] + df['charging_stations_near_work'] + 1)
    )

    # Desire × friction — high buy_score + high worry_score = conflicted buyer
    df['buy_x_worry'] = df['buy_score'] * df['worry_score']

    # Income × concern — motivated AND has money
    df['income_x_concern'] = df['annual_income_usd'] / 1e5 * df['environmental_concern_level']

    # Age × income — life-stage wealth proxy
    df['age_x_income'] = df['age'] * df['annual_income_usd'] / 1e7

    return df


train = build_interactions(train)
test  = build_interactions(test)

print("7 interaction features added")


## 5. Categorical Encoding

Three strategies depending on the feature type:

- **Ordinal encoding** — for features with a natural order (`range_anxiety_level`, `city_type`)
- **Binary encoding** — for Yes/No flags (`subsidy_available`, `home_charging_possible`)
- **Label encoding** — for nominal categories with no order (`gender`, `current_car_type`)

Train and test are encoded together to ensure consistent label mapping.


In [ ]:
def encode_categoricals(train_df, test_df):
    train_df = train_df.copy()
    test_df  = test_df.copy()

    # Ordinal: preserve meaningful rank order
    ordinal_maps = {
        'range_anxiety_level': {'Low': 0, 'Medium': 1, 'High': 2},
        'city_type':           {'Rural': 0, 'Suburban': 1, 'Urban': 2},
    }
    for col, mapping in ordinal_maps.items():
        train_df[col] = train_df[col].astype(str).map(mapping)
        test_df[col]  = test_df[col].astype(str).map(mapping)

    # Binary: Yes → 1, No → 0
    for col in ['subsidy_available', 'home_charging_possible']:
        train_df[col] = (train_df[col].astype(str) == 'Yes').astype(int)
        test_df[col]  = (test_df[col].astype(str) == 'Yes').astype(int)

    # Label encoding: fit on combined train+test to avoid unseen category errors
    for col in ['gender', 'current_car_type']:
        le = LabelEncoder()
        le.fit(pd.concat([train_df[col], test_df[col]]).astype(str))
        train_df[col] = le.transform(train_df[col].astype(str))
        test_df[col]  = le.transform(test_df[col].astype(str))

    return train_df, test_df


train, test = encode_categoricals(train, test)

print("Categorical encoding complete")
print(f"  range_anxiety_level: {sorted(train['range_anxiety_level'].unique())}")
print(f"  city_type:           {sorted(train['city_type'].unique())}")
print(f"  subsidy_available:   {sorted(train['subsidy_available'].unique())}")


## 6. Assemble and Save

Select all 22 features, save to `data/processed/` for use in model training.
Target `y` is saved separately so it can be loaded alongside features without
having to re-parse the raw CSV.


In [ ]:
FEATURE_COLS = [
    # Numeric (raw)
    'age', 'annual_income_usd', 'daily_commute_km',
    'number_of_cars_owned', 'charging_stations_near_home',
    'charging_stations_near_work', 'environmental_concern_level',
    # Categorical (encoded)
    'gender', 'city_type', 'current_car_type',
    'home_charging_possible', 'subsidy_available', 'range_anxiety_level',
    # Manual scores
    'buy_score', 'worry_score',
    # Interaction features
    'income_x_subsidy', 'concern_x_subsidy', 'income_per_car',
    'commute_per_charger', 'buy_x_worry', 'income_x_concern', 'age_x_income',
]

X      = train[FEATURE_COLS].copy()
X_test = test[FEATURE_COLS].copy()

PROC.mkdir(parents=True, exist_ok=True)
X.to_csv(PROC / 'X_train.csv', index=False)
X_test.to_csv(PROC / 'X_test.csv', index=False)
y.to_frame().to_csv(PROC / 'y_train.csv', index=False)

print(f"Total features: {len(FEATURE_COLS)}")
print(f"Train: {X.shape},  Test: {X_test.shape}")
print()
print("Saved:")
print(f"  {PROC}/X_train.csv")
print(f"  {PROC}/X_test.csv")
print(f"  {PROC}/y_train.csv")


## 7. Summary

All features are built and saved. No missing values — all columns are numeric and ready for LightGBM.

| Group | Features | Notes |
|---|---|---|
| Numeric (raw) | `age`, `annual_income_usd`, `daily_commute_km`, `number_of_cars_owned`, `charging_stations_near_home`, `charging_stations_near_work`, `environmental_concern_level` | Used as-is |
| Categorical | `gender`, `city_type`, `current_car_type`, `home_charging_possible`, `subsidy_available`, `range_anxiety_level` | Ordinal / binary / label encoded |
| Manual scores | `buy_score`, `worry_score` | Derived from EDA coefficients |
| Interactions | `income_x_subsidy`, `concern_x_subsidy`, `income_per_car`, `commute_per_charger`, `buy_x_worry`, `income_x_concern`, `age_x_income` | Cross-products of key variables |

**Next step:** `baseline.ipynb` — model training with nested target encoding and LightGBM.
